In [0]:
from pyspark.sql.functions import current_timestamp, lit, col

In [0]:
%sql
CREATE EXTERNAL VOLUME IF NOT EXISTS alwaha_banking_dev_001.bronze.landing_swipes_volume
LOCATION 'abfss://landing@alwahabankingdev001.dfs.core.windows.net/stream_card_swipes/';

In [0]:
dbutils.widgets.text("p_adf_run_id", "default_run_id")
adf_run_id = dbutils.widgets.get("p_adf_run_id")

default_options ={
        "mergeSchema" : "true",
        "tblproperties.delta.autoOptimize.optimizeWrite" : "true",
        "tblproperties.delta.autoOptimize.autoCompact" : "true",
        "tblproperties.delta.enableChildFileCompression" : "true"
    }

landing_swipe_path = f"/Volumes/alwaha_banking_dev_001/bronze/landing_swipes_volume/"
schema_path = f"/Volumes/alwaha_banking_dev_001/bronze/raw_files_volume/_schemas/stream_card_swipes"
check_point_path = f"abfss://bronze@alwahabankingdev001.dfs.core.windows.net/deltatables/_checkpoints/v3/stream_card_swipes"

df = (spark.readStream.format("cloudFiles")
      .option("cloudFiles.format", "json")
      .option("multiline", "false")
      .option("cloudFiles.schemaLocation", schema_path)
      .option("pathGlobFilter", "*.jsonl")
      .load(landing_swipe_path)
      )

df_finnal = (df.withColumns({
    "ingested_at": current_timestamp(),
    "file_name": col("_metadata.file_path"),
    "adf_run_id" : lit(adf_run_id)
}))

query = (df_finnal.writeStream
         .format("delta")
         .option("checkpointLocation", check_point_path)
         .options(**default_options)
         .outputMode("append")
         .trigger(availableNow=True)
         .toTable("alwaha_banking_dev_001.bronze.stream_card_swipes")
        )
query.awaitTermination()

try:
    spark.sql("""
              ALTER TABLE alwaha_banking_dev_001.bronze.stream_card_swipes 
              CLUSTER BY (card_number, timestamp)""")
    
except Exception as e:
    print(f"⚠️ Clustering Note: {str(e)}")